## Remember Multipart CIFAR-10 and Run Mock Training

### Objective
Recover the manifest from S3, remember all CIFAR archive parts from S3, reconstruct the full archive, decode a CIFAR-10 subset, and run a tiny mock PyTorch training loop.

In [ ]:
%load_ext autoreload
%autoreload 2

### Objective
Import the tools needed for reconstruction, dataset extraction, and the simple training loop.

In [ ]:
import hashlib
import io
import json
import pickle
import zipfile

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import laila
from laila.pool import S3Pool

POOL_NICKNAME = "cifar_10_s3_pool"
MANIFEST_NICKNAME = "cifar_10_s3_manifest"
BATCH_SIZE = 64
MAX_SAMPLES = 512
MAX_STEPS = 5

### Objective
Define the tiny model and helper functions for loading the manifest, rebuilding the archive bytes, and extracting a CIFAR subset from the reconstructed zip.

In [ ]:
class TinyCIFARNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )


def manifest_entry_id_from_nickname(manifest_nickname: str) -> str:
    # Nicknames deterministically map to entry IDs, so the manifest can be re-fetched from S3.
    return laila.constant(data=np.zeros(1, dtype=np.uint8), nickname=manifest_nickname).global_id


def entry_payload_to_bytes(payload: object) -> bytes:
    # Every remembered chunk was stored as uint8 data in the upload notebook.
    return np.asarray(payload, dtype=np.uint8).tobytes()


def load_manifest_from_s3(manifest_entry_id: str, pool_nickname: str) -> dict:
    manifest_future = laila.remember(manifest_entry_id, pool_nickname=pool_nickname)
    print("Manifest future type:", type(manifest_future).__name__)
    manifest_future.wait()
    manifest_entry = manifest_future.result
    return dict(manifest_entry.data)


def reconstruct_archive_bytes(manifest: dict, recovered_entries: list) -> bytes:
    # Reassemble the archive using the manifest's original part ordering.
    recovered_by_id = {entry.global_id: entry for entry in recovered_entries}
    ordered_chunks = [
        entry_payload_to_bytes(recovered_by_id[entry_id].data)
        for entry_id in manifest["entry_ids"]
    ]
    archive_bytes = b"".join(ordered_chunks)

    # Validate the reconstruction against the upload-time hash.
    archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
    if archive_sha256 != manifest["archive_sha256"]:
        raise RuntimeError(
            "Reconstructed archive hash mismatch: "
            f"{archive_sha256} != {manifest['archive_sha256']}"
        )

    return archive_bytes


def load_cifar_subset_from_zip_bytes(zip_bytes: bytes, max_samples: int) -> tuple[torch.Tensor, torch.Tensor]:
    # Extract a small subset from the CIFAR training batches for a quick demo loop.
    images = []
    labels = []

    with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as zf:
        batch_names = sorted(
            name
            for name in zf.namelist()
            if "data_batch_" in name and not name.endswith("/")
        )
        for batch_name in batch_names:
            with zf.open(batch_name, "r") as f:
                batch = pickle.load(f, encoding="bytes")
            batch_images = batch[b"data"].reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
            batch_labels = np.asarray(batch[b"labels"], dtype=np.int64)
            images.append(batch_images)
            labels.append(batch_labels)
            if sum(len(x) for x in labels) >= max_samples:
                break

    if not images:
        raise RuntimeError("No CIFAR-10 data_batch files were found inside the reconstructed zip archive.")

    image_tensor = torch.from_numpy(np.concatenate(images, axis=0)[:max_samples])
    label_tensor = torch.from_numpy(np.concatenate(labels, axis=0)[:max_samples])
    return image_tensor, label_tensor

### Objective
Register the S3 pool, derive the manifest entry ID from its deterministic nickname, and recover the manifest directly from S3.

In [ ]:
pool_nickname = POOL_NICKNAME
manifest_entry_id = manifest_entry_id_from_nickname(MANIFEST_NICKNAME)

s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=pool_nickname)

manifest = load_manifest_from_s3(manifest_entry_id, pool_nickname=pool_nickname)

print("Manifest entry id:", manifest_entry_id)
print("Part count:", manifest["num_parts"])
print("Pool nickname:", pool_nickname)

### Objective
Fetch all archive parts with `laila.remember(...)`, wait on the resulting `GroupFuture`, and collect the recovered entries.

In [ ]:
# Passing a list of entry ids gives back a GroupFuture with one child future per part.
fetch_group_future = laila.remember(manifest["entry_ids"], pool_nickname=pool_nickname)
print("Remember future type:", type(fetch_group_future).__name__)
print("Status before wait:", fetch_group_future.status)

fetch_group_future.wait()
print("Status after wait:", fetch_group_future.status)

# After waiting, each child future should hold one recovered LAILA entry.
recovered_entries = [child_future.result for child_future in fetch_group_future]
print("Recovered entries:", len(recovered_entries))

### Objective
Reconstruct the original archive bytes, verify integrity, and decode a small CIFAR-10 training subset into tensors.

In [ ]:
archive_bytes = reconstruct_archive_bytes(manifest, recovered_entries)
images, labels = load_cifar_subset_from_zip_bytes(archive_bytes, max_samples=MAX_SAMPLES)

print("Archive bytes:", len(archive_bytes))
print("Images shape:", tuple(images.shape))
print("Labels shape:", tuple(labels.shape))

### Objective
Create a `DataLoader`, define a tiny CNN, and run a short mock training loop that proves the reconstructed archive can feed a PyTorch workflow.

In [ ]:
loader = DataLoader(TensorDataset(images, labels), batch_size=BATCH_SIZE, shuffle=True)
model = TinyCIFARNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

model.train()
for step, (batch_images, batch_labels) in enumerate(loader, start=1):
    # Standard training step: clear grads, run forward pass, backprop, update.
    optimizer.zero_grad()
    logits = model(batch_images)
    loss = criterion(logits, batch_labels)
    loss.backward()
    optimizer.step()

    print(
        f"step={step:02d} "
        f"batch_shape={tuple(batch_images.shape)} "
        f"loss={loss.item():.4f}"
    )
    if step >= MAX_STEPS:
        break

print("Mock CIFAR training complete.")